# **# BIBLIOTECAS E MÓDULOS**

In [1]:

import heapq
import numpy as np
import pandas as pd
import openpyxl as op
from openpyxl import load_workbook

import os


ARQ = "0_5_5_CARTEIRA_FM_DADOS_CLOSE.xlsx"
N_ATIVOS = 30

# **# RETORNOS MENSAIS E ANUAIS - CARTEIRA FM**

In [2]:

# ----------------------- CARGA & LIMPEZA -----------------------

def carregar_base(caminho, sheet=None):
    """Lê a planilha e retorna preços em formato largo (índice=Date, colunas=tickers)."""
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # Localiza e converte a coluna de datas
    col_date = "Date" if "Date" in df.columns else None
    if col_date is None:
        cand = [c for c in df.columns if c.lower() in ("date", "data", "dt")]
        col_date = cand[0] if cand else None
    if col_date is None:
        raise ValueError("Coluna de data não encontrada (ex.: 'Date').")

    if np.issubdtype(df[col_date].dtype, np.number):
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30")
    else:
        df["Date"] = pd.to_datetime(df[col_date])

    # Ordena, remove duplicadas de data e colunas totalmente vazias
    df = df.drop_duplicates(subset=["Date"]).sort_values("Date")
    for c in list(df.columns):
        if c == "Date":
            continue
        if df[c].isna().all():
            df.drop(columns=[c], inplace=True)

    # Constrói a tabela larga
    tickers = [c for c in df.columns if c != "Date"]
    if len(tickers) < N_ATIVOS:
        raise ValueError(f"Foram encontrados {len(tickers)} ativos (< {N_ATIVOS}).")

    tickers = tickers[:N_ATIVOS]  # garante 30 primeiros
    wide = df[["Date"] + tickers].set_index("Date")
    wide = wide.apply(pd.to_numeric, errors="coerce").ffill().bfill()
    return wide[tickers], tickers

# ----------------------- NUCLEO DO ALGORITMO -----------------------

def nivelar_minimo(precos: pd.Series):
    """
    Determina o aporte inicial mínimo para 'nivelar' os 30 papéis o mais próximo possível, com inteiros.
    Estratégia:
      - Começa com 1 cota de cada papel (garante todos presentes).
      - Em cada passo, compra +1 do papel com menor valor investido (v_i = q_i * p_i),
        DESDE QUE isso não ultrapasse o atual valor máximo (para, então, no 1º ponto
        em que qualquer novo incremento de quem está no mínimo passaria o máximo).
    Retorna: (qtd_por_ticker: pd.Series, capital_inicial_minimo: float)
    """
    p = precos.astype(float).copy()
    tickers = p.index.tolist()
    n = len(p)

    # começamos com 1 unidade de cada (todos presentes)
    q = np.ones(n, dtype=int)
    v = q * p.values

    # heap min por valor investido; cada item: (v_i, i)
    heap = [(v[i], i) for i in range(n)]
    heapq.heapify(heap)
    v_max = float(v.max())

    # incremento discreto (passo) de cada papel é p_i
    while True:
        v_min, i = heapq.heappop(heap)
        passo = p.iloc[i]

        # Regra de parada: se elevar o mínimo ultrapassa o máximo atual, paramos.
        if v_min + passo > v_max:
            # devolve e encerra
            heapq.heappush(heap, (v_min, i))
            break

        # Caso contrário, comprar +1 do papel i
        q[i] += 1
        v_min_novo = v_min + passo
        v_max = max(v_max, v_min_novo)
        heapq.heappush(heap, (v_min_novo, i))

    q = pd.Series(q, index=tickers, name="Qtd")
    capital = float((q * p).sum())
    return q, capital

def equalizar_com_orcamento(precos: pd.Series, capital: float):
    """
    Equaliza com inteiros (o mais próximo possível) dado um orçamento fixo.
    1) floor(alvo/preço) por papel
    2) usa o caixa restante para comprar 1 unidade por vez do papel com MAIOR déficit
       (deficit_i = alvo - investido_i), enquanto houver saldo suficiente.
    Retorna: (qtd_por_ticker: pd.Series, caixa: float)
    """
    p = precos.astype(float).copy()
    tickers = p.index.tolist()
    n = len(p)

    alvo = capital / n
    q = np.floor(alvo / p).astype(int).clip(lower=0)
    invest = q * p
    gasto = float(invest.sum())
    caixa = float(capital - gasto)

    # Se por algum motivo (preços muito altos) algum ativo ficou zerado, tudo bem:
    # a etapa 2) vai tentar completar onde couber.

    # loop de alocação do caixa por "déficit"
    while True:
        invest = q * p
        deficit = alvo - invest
        # seleciona o papel com maior déficit positivo e cujo preço caiba no caixa
        candidatos = deficit[deficit > 0].sort_values(ascending=False)
        if candidatos.empty:
            break
        comprado = False
        for idx in candidatos.index:
            preco = p.loc[idx]
            if caixa >= preco:
                q.loc[idx] += 1
                caixa -= preco
                comprado = True
                break
        if not comprado:
            break  # caixa insuficiente para qualquer compra

    return q.astype(int), float(caixa)

# ----------------------- SIMULAÇÃO ANUAL -----------------------

def primeiros_pregoes_por_ano(index_datas: pd.DatetimeIndex):
    anos = sorted(set(index_datas.year))
    dts = []
    for a in anos:
        f = index_datas[index_datas.year == a]
        if len(f) > 0:
            dts.append(f.min())
    return dts

def simular(wide: pd.DataFrame):
    """
    1) Determina o aporte inicial mínimo (nivelamento discreto) no 1º pregão do 1º ano.
    2) Para cada ano subsequente, rebalanceia no 1º pregão com o valor de mercado do dia.
    Retorna:
      - serie_valor: série diária do valor total da carteira,
      - ret_mensal_df: DataFrame com retornos mensais,
      - ret_anual_df: DataFrame com retornos anuais,
      - aloc_inicial: (DataFrame alocação inicial, data, capital_inicial)
    """
    idx = wide.index
    dts_reb = primeiros_pregoes_por_ano(idx)

    serie_valor = pd.Series(index=idx, dtype=float)

    # --- 1) ALOC. INICIAL (apo rte mínimo) ---
    d0 = dts_reb[0]
    precos0 = wide.loc[d0]
    q0, capital0 = nivelar_minimo(precos0)  # aporte inicial mínimo
    caixa = 0.0  # no ponto inicial, usamos exatamente o capital mínimo (sem sobra)

    # Preenche o valor diário do primeiro bloco com q0 fixas
    dt_fim0 = (dts_reb[1] - pd.Timedelta(days=1)) if len(dts_reb) > 1 else idx[-1]
    bloco0 = wide.loc[(idx >= d0) & (idx <= dt_fim0)]
    serie_valor.loc[bloco0.index] = (bloco0 * q0).sum(axis=1) + caixa

    # --- 2) REBALANCEAMENTOS ANUAIS ---
    q = q0.copy()
    for i in range(1, len(dts_reb)):
        dt_reb = dts_reb[i]
        dt_fim = (dts_reb[i+1] - pd.Timedelta(days=1)) if i+1 < len(dts_reb) else idx[-1]
        bloco = wide.loc[(idx >= dt_reb) & (idx <= dt_fim)]
        if bloco.empty:
            continue

        # Valor de mercado no 1º dia do bloco
        capital = float((q * bloco.iloc[0]).sum() + caixa)

        # Equaliza com orçamento (inteiros)
        q, caixa = equalizar_com_orcamento(bloco.iloc[0], capital)

        # Valor diário no bloco
        serie_valor.loc[bloco.index] = (bloco * q).sum(axis=1) + caixa

    # --- Tabelas de retorno ---
    # Mensal (último dia útil do mês)
    vm = serie_valor.resample("ME").last().dropna()
    ret_mensal = vm.pct_change().dropna()
    ret_mensal_df = (ret_mensal.to_frame("Retorno Mensal FM")
                     .assign(Ano=lambda d: d.index.year, Mes=lambda d: d.index.month)
                     .loc[:, ["Ano", "Mes", "Retorno Mensal FM"]])

    # Anual (último dia útil do ano - dezembro)
    va = serie_valor.resample("YE-DEC").last().dropna()
    ret_anual = va.pct_change().dropna()
    ret_anual.index = ret_anual.index.year
    ret_anual_df = ret_anual.to_frame("Retorno Anual_FM")

    # Tabela de alocação inicial
    aloc = pd.DataFrame({
        "Ticker": q0.index,
        "Preço": precos0.values,
        "Qtd (inteira)": q0.values,
        "Investido": (q0.values * precos0.values)
    }).sort_values("Ticker").reset_index(drop=True)

    return serie_valor, ret_mensal_df, ret_anual_df, (aloc, d0, capital0)

# ----------------------- EXECUÇÃO -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    wide, tickers = carregar_base(ARQ)
    serie_valor, ret_mensal_df, ret_anual_df, aloc_inicial = simular(wide)

    aloc, d0, capital0 = aloc_inicial

    print("\n=== REBALANCEAMENTO MÍNIMO (inteiros) ===")
    print(f"Período na base: {wide.index.min().date()} até {wide.index.max().date()} ({len(wide)} pregões)")
    print(f"Número de ativos: {len(wide.columns)}")
    print(f"1º rebalanceamento: {pd.to_datetime(d0).date()}")

    print("\n--- Alocação inicial (aporte MÍNIMO determinado) ---")
    print(aloc.to_string(index=False))
    print(f"Aporte inicial mínimo (R$): {capital0:,.2f}")

    print("\n--- Retornos ANUAIS ---")
    print(ret_anual_df.to_string())

    print("\n--- Retornos MENSAIS (primeiras 24 linhas) ---")
    print(ret_mensal_df.head(24).to_string(index=False))
    print("...")
    print("\n--- Retornos MENSAIS (últimas 24 linhas) ---")
    print(ret_mensal_df.tail(24).to_string(index=False))

    print("\nValor final da carteira (R$): {:.2f}".format(float(serie_valor.dropna().iloc[-1])))



=== REBALANCEAMENTO MÍNIMO (inteiros) ===
Período na base: 2011-01-03 até 2019-12-30 (2233 pregões)
Número de ativos: 30
1º rebalanceamento: 2011-01-03

--- Alocação inicial (aporte MÍNIMO determinado) ---
  Ticker      Preço  Qtd (inteira)  Investido
AHEB3.SA  25.891182              6 155.347092
AHEB5.SA  28.545076              5 142.725382
AHEB6.SA  14.844518             10 148.445177
BMKS3.SA 161.604324              1 161.604324
BRAP3.SA   2.948866             49 144.494417
BRAP4.SA   3.129102             46 143.938681
BRKM3.SA   9.669642             15 145.044637
BRKM5.SA  11.340091             13 147.421180
BRKM6.SA   7.462964             20 149.259281
CAMB3.SA   7.832460             19 148.816748
EALT3.SA   5.038043             29 146.103234
EALT4.SA   1.884280             76 143.205314
ECOR3.SA   7.554218             19 143.530148
FESA3.SA   1.682022             85 142.971898
FESA4.SA   1.298621            110 142.848330
GRND3.SA   0.965708            148 142.924822
GUAR3.SA   

# **# REBALANCEAMENTO DA CARTEIRA FM EM QUANTIDADE DE CADA PAPEL NO INÍCIO DE CADA ANO**

In [3]:
def simular_com_quantidades(wide: pd.DataFrame):
    idx = wide.index
    dts_reb = primeiros_pregoes_por_ano(idx)
    
    # Dicionário para armazenar as quantidades de cada ano
    historico_qtds = {}

    # --- 1) ALOC. INICIAL (Aporte Mínimo) ---
    d0 = dts_reb[0]
    precos0 = wide.loc[d0]
    q, capital0 = nivelar_minimo(precos0)
    caixa = 0.0
    
    historico_qtds[d0.year] = q.copy()

    # --- 2) REBALANCEAMENTOS ANUAIS ---
    for i in range(1, len(dts_reb)):
        dt_reb = dts_reb[i]
        precos_atual = wide.loc[dt_reb]
        
        # Valor total da carteira no momento do rebalanceamento
        valor_mercado = (q * precos_atual).sum() + caixa
        
        # Novo rebalanceamento (Equalização com Orçamento)
        q, caixa = equalizar_com_orcamento(precos_atual, valor_mercado)
        
        # Salva as quantidades do novo ano
        historico_qtds[dt_reb.year] = q.copy()

    # Transformar o dicionário em DataFrame (Ativos x Anos)
    df_quantidades = pd.DataFrame(historico_qtds)
    df_quantidades.index.name = 'Ticker'
    
    return df_quantidades

# Execução
wide_data, _ = carregar_base(ARQ)
df_resumo_qtds = simular_com_quantidades(wide_data)

print("=== QUANTIDADES DE PAPÉIS POR ANO (REBALANCEAMENTO) ===")
display(df_resumo_qtds)

=== QUANTIDADES DE PAPÉIS POR ANO (REBALANCEAMENTO) ===


,2011,2012,2013,2014,2015,2016,2017,2018,2019
Ticker,,,,,,,,,
BMKS3.SA,1,2,2,3,3,3,3,4,5
GRND3.SA,148,172,93,95,103,81,125,120,142
BRKM5.SA,13,18,22,16,18,10,13,15,14
BRKM3.SA,15,22,29,22,31,18,15,15,14
WLMM3.SA,32,31,39,45,43,39,69,96,152
WLMM4.SA,28,23,22,26,23,59,174,143,150
PNVL3.SA,57,48,23,35,33,22,17,42,59
AHEB3.SA,6,12,10,14,13,13,7,5,7
AHEB6.SA,10,6,7,8,8,24,7,8,9


# **# REBALANCEAMENTO DA CARTEIRA FM EM VALOR FINANCEIRO DE CADA PAPEL NO INÍCIO DE CADA ANO**

In [4]:
def simular_valores_financeiros_v2(wide: pd.DataFrame):
    idx = wide.index
    dts_reb = primeiros_pregoes_por_ano(idx)
    historico_valores = {}

    # --- 1) ALOC. INICIAL ---
    d0 = dts_reb[0]
    precos0 = wide.loc[d0]
    q, capital0 = nivelar_minimo(precos0)
    caixa = 0.0
    
    # Valor por papel e Total
    valores_ano = q * precos0
    valores_ano['VALOR TOTAL'] = valores_ano.sum() + caixa
    historico_valores[d0.year] = valores_ano

    # --- 2) REBALANCEAMENTOS ANUAIS ---
    for i in range(1, len(dts_reb)):
        dt_reb = dts_reb[i]
        precos_atual = wide.loc[dt_reb]
        valor_total_antes = (q * precos_atual).sum() + caixa
        
        # Rebalanceamento
        q, caixa = equalizar_com_orcamento(precos_atual, valor_total_antes)
        
        # Salva valores
        valores_ano = q * precos_atual
        valores_ano['VALOR TOTAL'] = valores_ano.sum() + caixa
        historico_valores[dt_reb.year] = valores_ano

    # Consolidar
    df_valores = pd.DataFrame(historico_valores)
    
    # Inserir linha vazia antes do VALOR TOTAL
    tickers = [t for t in df_valores.index if t != 'VALOR TOTAL']
    df_final = pd.concat([
        df_valores.loc[tickers], 
        pd.DataFrame(index=[' '], columns=df_valores.columns), 
        df_valores.loc[['VALOR TOTAL']]
    ])
    
    df_final.index.name = 'Ticker'
    return df_final

# Execução
wide_data, _ = carregar_base(ARQ)
df_resumo_financeiro = simular_valores_financeiros_v2(wide_data)

print("=== VALOR FINANCEIRO POR PAPEL E TOTAL POR ANO (R$) ===")
display(df_resumo_financeiro.style.format("{:.2f}", na_rep=""))

C:\Users\b48k\AppData\Local\Temp\ipykernel_38180\1361667109.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([


=== VALOR FINANCEIRO POR PAPEL E TOTAL POR ANO (R$) ===


,2011,2012,2013,2014,2015,2016,2017,2018,2019
Ticker,,,,,,,,,
BMKS3.SA,161.60,155.93,211.42,291.24,259.71,226.98,322.82,604.27,644.83
GRND3.SA,142.92,144.49,178.65,206.96,198.36,181.56,317.23,507.58,541.00
BRKM5.SA,147.42,143.31,181.90,198.24,197.62,179.94,326.17,499.37,559.60
BRKM3.SA,145.04,146.88,176.20,206.24,198.77,183.27,324.26,496.12,530.26
WLMM3.SA,146.20,141.64,178.19,205.60,196.46,178.19,315.25,507.15,542.55
WLMM4.SA,147.03,143.16,176.40,204.59,195.57,180.24,315.73,507.81,540.11
PNVL3.SA,144.59,144.31,176.01,201.74,198.28,174.77,313.12,501.36,534.24
AHEB3.SA,155.35,142.03,192.33,201.95,187.53,187.43,310.69,503.03,522.41
AHEB6.SA,148.45,155.25,181.17,207.05,207.05,177.42,310.49,473.13,532.27


# **# VOLATILIDADE ANUALIZADA - CARTEIRA FM**

In [5]:
# =========================================
# Volatilidade anualizada (a partir dos dados do script anterior)
# - Tenta ler 'rebalance_minimo_2010_2019.xlsx' (aba 'Serie_Valor').
# - Caso não exista, recalcula a série diária replicando o rebalanceamento mínimo anual.
# - Saída: volatilidade anual e do período, por base diária e mensal.
# =========================================

ARQ_SAIDA_ANTERIOR = 'rebalance_minimo_2010_2019.xlsx'  # Excel gerado pelo script anterior
ABA_SERIE = 'Serie_Valor'
ARQ_ORIGEM = '0_5_5_CARTEIRA_FM_DADOS_CLOSE.xlsx'   # Backup: reconstrói a curva se necessário
N_ATIVOS = 30

# ----------- Utilidades (iguais ao script anterior, para fallback) -----------

def carregar_base(caminho, sheet=None):
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine='openpyxl')
    df.columns = [str(c).strip() for c in df.columns]

    # Coluna de data
    col_date = 'Date' if 'Date' in df.columns else None
    if col_date is None:
        cand = [c for c in df.columns if c.lower() in ('date','data','dt')]
        col_date = cand[0] if cand else None
    if col_date is None:
        raise ValueError("Coluna 'Date' não encontrada.")

    # Converte datas
    if np.issubdtype(df[col_date].dtype, np.number):
        df['Date'] = pd.to_datetime(df[col_date], unit='d', origin='1899-12-30')
    else:
        df['Date'] = pd.to_datetime(df[col_date])

    # Ordena e limpa
    df = df.drop_duplicates(subset=['Date']).sort_values('Date')
    for c in list(df.columns):
        if c == 'Date':
            continue
        if df[c].isna().all():
            df.drop(columns=[c], inplace=True)

    tickers = [c for c in df.columns if c != 'Date']
    if len(tickers) < N_ATIVOS:
        raise ValueError(f"Foram encontrados {len(tickers)} ativos (< {N_ATIVOS}).")

    tickers = tickers[:N_ATIVOS]
    wide = df[['Date'] + tickers].set_index('Date')
    wide = wide.apply(pd.to_numeric, errors='coerce').ffill().bfill()
    return wide[tickers], tickers

def primeiros_pregoes_por_ano(index_datas: pd.DatetimeIndex):
    anos = sorted(set(index_datas.year))
    return [index_datas[index_datas.year == a].min() for a in anos if (index_datas.year == a).any()]

def nivelar_minimo(precos: pd.Series):
    import heapq
    p = precos.astype(float).copy()
    n = len(p)
    q = np.ones(n, dtype=int)
    v = q * p.values
    heap = [(v[i], i) for i in range(n)]
    heapq.heapify(heap)
    v_max = float(v.max())
    while True:
        v_min, i = heapq.heappop(heap)
        passo = p.iloc[i]
        if v_min + passo > v_max:
            heapq.heappush(heap, (v_min, i))
            break
        q[i] += 1
        v_min_novo = v_min + passo
        v_max = max(v_max, v_min_novo)
        heapq.heappush(heap, (v_min_novo, i))
    q = pd.Series(q, index=p.index)
    capital = float((q * p).sum())
    return q, capital

def equalizar_com_orcamento(precos: pd.Series, capital: float):
    p = precos.astype(float)
    n = len(p)
    alvo = capital / n
    q = np.floor(alvo / p).astype(int).clip(lower=0)
    invest = q * p
    caixa = float(capital - float(invest.sum()))
    # gasta o caixa por déficit
    while True:
        invest = q * p
        deficit = alvo - invest
        cand = deficit[deficit > 0].sort_values(ascending=False)
        if cand.empty:
            break
        comprado = False
        for idx in cand.index:
            if caixa >= p.loc[idx]:
                q.loc[idx] += 1
                caixa -= float(p.loc[idx])
                comprado = True
                break
        if not comprado:
            break
    return q.astype(int), float(caixa)

def reconstruir_serie_valor(arquivo_origem):
    wide, _ = carregar_base(arquivo_origem)
    idx = wide.index
    datas_reb = primeiros_pregoes_por_ano(idx)

    # Alocação inicial: aporte mínimo
    d0 = datas_reb[0]
    precos0 = wide.loc[d0]
    q, capital0 = nivelar_minimo(precos0)
    caixa = 0.0

    serie_valor = pd.Series(index=idx, dtype=float)
    # primeiro bloco
    dt_fim0 = (datas_reb[1] - pd.Timedelta(days=1)) if len(datas_reb) > 1 else idx[-1]
    bloco0 = wide.loc[(idx >= d0) & (idx <= dt_fim0)]
    serie_valor.loc[bloco0.index] = (bloco0 * q).sum(axis=1) + caixa

    # rebalanceamentos seguintes
    for i in range(1, len(datas_reb)):
        dt_reb = datas_reb[i]
        dt_fim = (datas_reb[i+1] - pd.Timedelta(days=1)) if i+1 < len(datas_reb) else idx[-1]
        bloco = wide.loc[(idx >= dt_reb) & (idx <= dt_fim)]
        capital = float((q * bloco.iloc[0]).sum() + caixa)
        q, caixa = equalizar_com_orcamento(bloco.iloc[0], capital)
        serie_valor.loc[bloco.index] = (bloco * q).sum(axis=1) + caixa

    return serie_valor

# ----------- Carrega a série diária do portfólio -----------

if os.path.exists(ARQ_SAIDA_ANTERIOR):
    # Lê a série do Excel de saída
    serie_valor = pd.read_excel(ARQ_SAIDA_ANTERIOR, sheet_name=ABA_SERIE, engine='openpyxl')
    serie_valor.columns = [str(c).strip() for c in serie_valor.columns]
    # Esperado: colunas ['Date', 'Valor_Portfolio']
    # Se 'Date' vier como índice no Excel, ajusta:
    if 'Date' in serie_valor.columns:
        serie_valor['Date'] = pd.to_datetime(serie_valor['Date'])
        serie_valor = serie_valor.set_index('Date')['Valor_Portfolio'].astype(float).sort_index()
    else:
        # Talvez já esteja com índice datetime
        serie_valor.index = pd.to_datetime(serie_valor.index)
        serie_valor = serie_valor.iloc[:, 0].astype(float).sort_index()
else:
    # Reconstrói a série caso o Excel não exista
    serie_valor = reconstruir_serie_valor(ARQ_ORIGEM)

# ----------- Cálculo de volatilidades -----------

# Retornos diários
ret_diario = serie_valor.pct_change().dropna()

# Retornos mensais (último útil do mês)
valor_mensal = serie_valor.resample('ME').last().dropna()
ret_mensal = valor_mensal.pct_change().dropna()

def vol_anualizada_diaria(serie_ret):
    """Desvio-padrão dos retornos diários × sqrt(252)."""
    return float(serie_ret.std(ddof=1) * np.sqrt(252))

def vol_anualizada_mensal(serie_ret):
    """Desvio-padrão dos retornos mensais × sqrt(12)."""
    return float(serie_ret.std(ddof=1) * np.sqrt(12))

# Vol para o período completo
vol_total_d = vol_anualizada_diaria(ret_diario)
vol_total_m = vol_anualizada_mensal(ret_mensal)

# Vol por ano (base diária)
vol_ano_d = []
for ano, sub in ret_diario.groupby(ret_diario.index.year):
    if len(sub) >= 2:
        vol_ano_d.append({'Ano': int(ano), 'Vol Anualizada (base diária)': vol_anualizada_diaria(sub)})
vol_ano_d = pd.DataFrame(vol_ano_d).sort_values('Ano')

# Vol por ano (base mensal)
vol_ano_m = []
for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
    if len(sub) >= 2:
        vol_ano_m.append({'Ano': int(ano), 'Vol Anualizada (base mensal)': vol_anualizada_mensal(sub)})
vol_ano_m = pd.DataFrame(vol_ano_m).sort_values('Ano')

# Junta as duas em uma tabela anual consolidada
vol_anual = pd.merge(vol_ano_d, vol_ano_m, on='Ano', how='outer').sort_values('Ano')

# ----------- Impressão -----------

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

print("\n=== VOLATILIDADE ANUALIZADA ===")
print(f"Período: {serie_valor.index.min().date()} a {serie_valor.index.max().date()}")
print(f"Nº de observações diárias: {len(serie_valor)}")

print("\n--- Por ano (anualizada) ---")
print(vol_anual.to_string(index=False))

print("\n--- Período completo ---")
print(f"Vol anualizada (base diária):  {vol_total_d:.2%}")
print(f"Vol anualizada (base mensal):  {vol_total_m:.2%}")


=== VOLATILIDADE ANUALIZADA ===
Período: 2011-01-03 a 2019-12-30
Nº de observações diárias: 2233

--- Por ano (anualizada) ---
 Ano  Vol Anualizada (base diária)  Vol Anualizada (base mensal)
2011                      0.112917                      0.082144
2012                      0.089752                      0.095180
2013                      0.172774                      0.116086
2014                      0.089272                      0.083209
2015                      0.117589                      0.130802
2016                      0.223970                      0.304344
2017                      0.115259                      0.105116
2018                      0.110553                      0.125991
2019                      0.240220                      0.259647

--- Período completo ---
Vol anualizada (base diária):  15.19%
Vol anualizada (base mensal):  17.42%


# **# VOLATILIDADE ANUALIZADA EM PORCENTAGEM - CARTEIRA FM**

In [6]:

ARQ_SAIDA_ANTERIOR = 'rebalance_minimo_2010_2019.xlsx'  # Excel gerado pelo script anterior
ABA_SERIE = 'Serie_Valor'
ARQ_ORIGEM = '2_1_DADOS_FM_2019_de_2010_2019_dt.xlsx'   # Backup: reconstrói a curva se necessário
N_ATIVOS = 30

# ----------- Utilidades (iguais ao script anterior, para fallback) -----------



def carregar_base(caminho, sheet=None):
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine='openpyxl')
    df.columns = [str(c).strip() for c in df.columns]

    # Coluna de data
    col_date = 'Date' if 'Date' in df.columns else None
    if col_date is None:
        cand = [c for c in df.columns if c.lower() in ('date','data','dt')]
        col_date = cand[0] if cand else None
    if col_date is None:
        raise ValueError("Coluna 'Date' não encontrada.")

    # Converte datas
    if np.issubdtype(df[col_date].dtype, np.number):
        df['Date'] = pd.to_datetime(df[col_date], unit='d', origin='1899-12-30')
    else:
        df['Date'] = pd.to_datetime(df[col_date])

    # Ordena e limpa
    df = df.drop_duplicates(subset=['Date']).sort_values('Date')
    for c in list(df.columns):
        if c == 'Date':
            continue
        if df[c].isna().all():
            df.drop(columns=[c], inplace=True)

    tickers = [c for c in df.columns if c != 'Date']
    if len(tickers) < N_ATIVOS:
        raise ValueError(f"Foram encontrados {len(tickers)} ativos (< {N_ATIVOS}).")

    tickers = tickers[:N_ATIVOS]
    wide = df[['Date'] + tickers].set_index('Date')
    wide = wide.apply(pd.to_numeric, errors='coerce').ffill().bfill()
    return wide[tickers], tickers

def primeiros_pregoes_por_ano(index_datas: pd.DatetimeIndex):
    anos = sorted(set(index_datas.year))
    return [index_datas[index_datas.year == a].min() for a in anos if (index_datas.year == a).any()]

def nivelar_minimo(precos: pd.Series):
    import heapq
    p = precos.astype(float).copy()
    n = len(p)
    q = np.ones(n, dtype=int)
    v = q * p.values
    heap = [(v[i], i) for i in range(n)]
    heapq.heapify(heap)
    v_max = float(v.max())
    while True:
        v_min, i = heapq.heappop(heap)
        passo = p.iloc[i]
        if v_min + passo > v_max:
            heapq.heappush(heap, (v_min, i))
            break
        q[i] += 1
        v_min_novo = v_min + passo
        v_max = max(v_max, v_min_novo)
        heapq.heappush(heap, (v_min_novo, i))
    q = pd.Series(q, index=p.index)
    capital = float((q * p).sum())
    return q, capital

def equalizar_com_orcamento(precos: pd.Series, capital: float):
    p = precos.astype(float)
    n = len(p)
    alvo = capital / n
    q = np.floor(alvo / p).astype(int).clip(lower=0)
    invest = q * p
    caixa = float(capital - float(invest.sum()))
    # gasta o caixa por déficit
    while True:
        invest = q * p
        deficit = alvo - invest
        cand = deficit[deficit > 0].sort_values(ascending=False)
        if cand.empty:
            break
        comprado = False
        for idx in cand.index:
            if caixa >= p.loc[idx]:
                q.loc[idx] += 1
                caixa -= float(p.loc[idx])
                comprado = True
                break
        if not comprado:
            break
    return q.astype(int), float(caixa)

def reconstruir_serie_valor(arquivo_origem):
    wide, _ = carregar_base(arquivo_origem)
    idx = wide.index
    datas_reb = primeiros_pregoes_por_ano(idx)

    # Alocação inicial: aporte mínimo
    d0 = datas_reb[0]
    precos0 = wide.loc[d0]
    q, capital0 = nivelar_minimo(precos0)
    caixa = 0.0

    serie_valor = pd.Series(index=idx, dtype=float)
    # primeiro bloco
    dt_fim0 = (datas_reb[1] - pd.Timedelta(days=1)) if len(datas_reb) > 1 else idx[-1]
    bloco0 = wide.loc[(idx >= d0) & (idx <= dt_fim0)]
    serie_valor.loc[bloco0.index] = (bloco0 * q).sum(axis=1) + caixa

    # rebalanceamentos seguintes
    for i in range(1, len(datas_reb)):
        dt_reb = datas_reb[i]
        dt_fim = (datas_reb[i+1] - pd.Timedelta(days=1)) if i+1 < len(datas_reb) else idx[-1]
        bloco = wide.loc[(idx >= dt_reb) & (idx <= dt_fim)]
        capital = float((q * bloco.iloc[0]).sum() + caixa)
        q, caixa = equalizar_com_orcamento(bloco.iloc[0], capital)
        serie_valor.loc[bloco.index] = (bloco * q).sum(axis=1) + caixa

    return serie_valor

# ----------- Carrega a série diária do portfólio -----------

if os.path.exists(ARQ_SAIDA_ANTERIOR):
    # Lê a série do Excel de saída
    serie_valor = pd.read_excel(ARQ_SAIDA_ANTERIOR, sheet_name=ABA_SERIE, engine='openpyxl')
    serie_valor.columns = [str(c).strip() for c in serie_valor.columns]
    # Esperado: colunas ['Date', 'Valor_Portfolio']
    if 'Date' in serie_valor.columns:
        serie_valor['Date'] = pd.to_datetime(serie_valor['Date'])
        serie_valor = serie_valor.set_index('Date')['Valor_Portfolio'].astype(float).sort_index()
    else:
        # Caso já venha com índice de datas
        serie_valor.index = pd.to_datetime(serie_valor.index)
        serie_valor = serie_valor.iloc[:, 0].astype(float).sort_index()
else:
    # Reconstrói a série caso o Excel não exista
    serie_valor = reconstruir_serie_valor(ARQ_ORIGEM)

# ----------- Cálculo de volatilidades -----------

# Retornos diários
ret_diario = serie_valor.pct_change().dropna()

# Retornos mensais (último dia útil do mês)
valor_mensal = serie_valor.resample('ME').last().dropna()
ret_mensal = valor_mensal.pct_change().dropna()

def vol_anualizada_diaria(serie_ret):
    """Desvio-padrão dos retornos diários × sqrt(252)."""
    return float(serie_ret.std(ddof=1) * np.sqrt(252))

def vol_anualizada_mensal(serie_ret):
    """Desvio-padrão dos retornos mensais × sqrt(12)."""
    return float(serie_ret.std(ddof=1) * np.sqrt(12))

# Vol para o período completo
vol_total_d = vol_anualizada_diaria(ret_diario)
vol_total_m = vol_anualizada_mensal(ret_mensal)

# Vol por ano (base diária)
vol_ano_d = []
for ano, sub in ret_diario.groupby(ret_diario.index.year):
    if len(sub) >= 2:
        vol_ano_d.append({'Ano': int(ano), 'Vol Anualizada (base diária)': vol_anualizada_diaria(sub)})
vol_ano_d = pd.DataFrame(vol_ano_d).sort_values('Ano')

# Vol por ano (base mensal)
vol_ano_m = []
for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
    if len(sub) >= 2:
        vol_ano_m.append({'Ano': int(ano), 'Vol Anualizada (base mensal)': vol_anualizada_mensal(sub)})
vol_ano_m = pd.DataFrame(vol_ano_m).sort_values('Ano')

# Junta as duas em uma tabela anual consolidada
vol_anual = pd.merge(vol_ano_d, vol_ano_m, on='Ano', how='outer').sort_values('Ano')

# ----------- Formatação em % (2 casas) -----------

def formatar_percent(df, cols):
    """Converte colunas para string em % com 2 casas decimais."""
    for c in cols:
        if c in df.columns:
            df[c] = (df[c] * 100).map(lambda x: f"{x:.2f}%")
    return df

vol_anual_fmt = vol_anual.copy()
vol_anual_fmt = formatar_percent(
    vol_anual_fmt,
    ['Vol Anualizada (base diária)', 'Vol Anualizada (base mensal)']
)

# ----------- Impressão -----------

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

print("\n=== VOLATILIDADE ANUALIZADA ===")
print(f"Período: {serie_valor.index.min().date()} a {serie_valor.index.max().date()}")
print(f"Nº de observações diárias: {len(serie_valor)}")

print("\n--- Por ano (anualizada) ---")
print(vol_anual_fmt.to_string(index=False))

print("\n--- Período completo ---")
print(f"Vol anualizada (base diária):  {vol_total_d*100:.2f}%")
print(f"Vol anualizada (base mensal):  {vol_total_m*100:.2f}%")


=== VOLATILIDADE ANUALIZADA ===
Período: 2011-01-03 a 2019-12-30
Nº de observações diárias: 2233

--- Por ano (anualizada) ---
 Ano Vol Anualizada (base diária) Vol Anualizada (base mensal)
2011                       11.29%                        8.21%
2012                        8.98%                        9.52%
2013                       17.28%                       11.61%
2014                        8.93%                        8.32%
2015                       11.76%                       13.08%
2016                       22.40%                       30.43%
2017                       11.53%                       10.51%
2018                       11.06%                       12.60%
2019                       24.02%                       25.96%

--- Período completo ---
Vol anualizada (base diária):  15.19%
Vol anualizada (base mensal):  17.42%


# **# DRAWDOWN MÁXIMO - CARTEIRA FM**

Abaixo está um complemento direto ao seu script de volatilidade para calcular e imprimir o Max Drawdown (MDD):

MDD do período completo (valor e datas de peak → trough).
MDD por ano-calendário.
Tudo em porcentagem com 2 casas decimais.


Definição: Drawdown em ttt = Vtmax⁡s≤tVs−1\frac{V_t}{\max_{s \le t} V_s} - 1maxs≤t​Vs​Vt​​−1.
O Max Drawdown é o menor (mais negativo) valor dessa série.



In [7]:
# ----------- Cálculo do Max Drawdown (MDD) -----------

def max_drawdown(serie_val):
    """
    Retorna:
      - mdd: float (ex.: -0.325 corresponde a -32,50%)
      - data_peak: data do pico que antecede o vale do MDD
      - data_trough: data do vale (onde o drawdown é máximo)
    """
    v = serie_val.astype(float).copy()
    roll_max = v.cummax()
    dd = (v / roll_max) - 1.0

    # MDD (valor mínimo)
    mdd = float(dd.min())

    # Datas de peak -> trough
    trough_idx = dd.idxmin()
    # até o trough, qual foi o pico?
    peak_idx = v.loc[:trough_idx].idxmax()

    return mdd, peak_idx, trough_idx, dd

# MDD do período completo
mdd_total, peak_total, trough_total, dd_series = max_drawdown(serie_valor)

# MDD por ano
mdd_por_ano = []
for ano, sub in serie_valor.groupby(serie_valor.index.year):
    if len(sub) >= 2:
        mdd_a, peak_a, trough_a, _ = max_drawdown(sub)
        mdd_por_ano.append({
            "Ano": int(ano),
            "MDD": mdd_a,
            "Peak": pd.to_datetime(peak_a).date(),
            "Trough": pd.to_datetime(trough_a).date()
        })
mdd_por_ano = pd.DataFrame(mdd_por_ano).sort_values("Ano", ascending=True)

# ----------- Impressão do MDD -----------

print("\n=== MAX DRAWDOWN (MDD) ===")

# Período completo
print(f"MDD do período: {mdd_total*100:.2f}%")
print(f"  Pico (peak)  : {pd.to_datetime(peak_total).date()}")
print(f"  Vale (trough): {pd.to_datetime(trough_total).date()}")

# Por ano (em % com 2 casas)
if not mdd_por_ano.empty:
    mdd_por_ano_fmt = mdd_por_ano.copy()
    mdd_por_ano_fmt["MDD"] = (mdd_por_ano_fmt["MDD"] * 100).map(lambda x: f"{x:.2f}%")
    print("\n--- MDD por ano ---")
    print(mdd_por_ano_fmt.to_string(index=False))


=== MAX DRAWDOWN (MDD) ===
MDD do período: -21.89%
  Pico (peak)  : 2014-07-08
  Vale (trough): 2016-01-26

--- MDD por ano ---
 Ano     MDD       Peak     Trough
2011 -13.35% 2011-07-04 2011-08-08
2012  -7.12% 2012-05-02 2012-06-11
2013 -12.10% 2013-05-08 2013-07-10
2014  -9.86% 2014-07-08 2014-12-15
2015 -15.74% 2015-05-04 2015-09-29
2016  -7.14% 2016-01-05 2016-01-26
2017  -4.32% 2017-05-16 2017-05-22
2018  -6.92% 2018-05-10 2018-06-19
2019  -8.12% 2019-09-24 2019-10-07


# **# ÍNDICE SHARPE - CARTEIRA FM**

Abaixo está um complemento direto para o script que você já tem (o de volatilidade e MDD). Este trecho calcula o Índice de Sharpe:

Sharpe anualizado (base diária) usando retornos diários do portfólio e um rate livre de risco anual (configurável; por padrão 0,00%).
Sharpe anualizado (base mensal) usando retornos mensais.
Sharpe por ano-calendário e para o período completo.


Fórmula (anualizado):
Sharpe=μ−rfσ\text{Sharpe} = \frac{\mu - r_f}{\sigma}Sharpe=σμ−rf​​
com μ\muμ e σ\sigmaσ sendo a média e o desvio-padrão dos excess returns (retorno da carteira menos o retorno livre de risco), e anualização via 252\sqrt{252}252​ (diário) ou 12\sqrt{12}12​ (mensal).

In [8]:
# ----------- Índice de Sharpe (anualizado) -----------

# Configuração: taxa livre de risco (ao ano, em decimal). Ajuste se desejar.
RISK_FREE_ANUAL = 0.00  # ex.: 0.065 para 6,5% a.a.

def rf_diario(rf_aa):
    """Converte risco livre anual para equivalente diário (252 pregões)."""
    return (1 + rf_aa)**(1/252) - 1

def rf_mensal(rf_aa):
    """Converte risco livre anual para equivalente mensal (12 meses)."""
    return (1 + rf_aa)**(1/12) - 1

def sharpe_anualizado_diario(ret_d, rf_aa):
    """Sharpe anualizado com base diária."""
    if len(ret_d) < 2:
        return np.nan
    rf_d = rf_diario(rf_aa)
    excess = ret_d - rf_d
    mu = excess.mean()
    sd = excess.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return np.nan
    return float((mu / sd) * np.sqrt(252))

def sharpe_anualizado_mensal(ret_m, rf_aa):
    """Sharpe anualizado com base mensal."""
    if len(ret_m) < 2:
        return np.nan
    rf_m = rf_mensal(rf_aa)
    excess = ret_m - rf_m
    mu = excess.mean()
    sd = excess.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return np.nan
    return float((mu / sd) * np.sqrt(12))

# Sharpe do período completo
sharpe_total_d = sharpe_anualizado_diario(ret_diario, RISK_FREE_ANUAL)
sharpe_total_m = sharpe_anualizado_mensal(ret_mensal, RISK_FREE_ANUAL)

# Sharpe por ano (base diária)
sharpe_ano_d = []
for ano, sub in ret_diario.groupby(ret_diario.index.year):
    s_ano = sharpe_anualizado_diario(sub, RISK_FREE_ANUAL)
    if not np.isnan(s_ano):
        sharpe_ano_d.append({"Ano": int(ano), "Sharpe (base diária)": s_ano})
sharpe_ano_d = pd.DataFrame(sharpe_ano_d).sort_values("Ano")

# Sharpe por ano (base mensal)
sharpe_ano_m = []
for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
    s_ano = sharpe_anualizado_mensal(sub, RISK_FREE_ANUAL)
    if not np.isnan(s_ano):
        sharpe_ano_m.append({"Ano": int(ano), "Sharpe (base mensal)": s_ano})
sharpe_ano_m = pd.DataFrame(sharpe_ano_m).sort_values("Ano")

# Consolida por ano
sharpe_anual = pd.merge(sharpe_ano_d, sharpe_ano_m, on="Ano", how="outer").sort_values("Ano")

# ----------- Impressão do Sharpe -----------

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

print("\n=== ÍNDICE DE SHARPE (ANUALIZADO) ===")
print(f"Taxa livre de risco (a.a.) considerada: {RISK_FREE_ANUAL*100:.2f}%")

# Tabela anual (formatada com 4 casas por ser uma razão adimensional)
if not sharpe_anual.empty:
    sharpe_anual_fmt = sharpe_anual.copy()
    for c in ["Sharpe (base diária)", "Sharpe (base mensal)"]:
        if c in sharpe_anual_fmt.columns:
            sharpe_anual_fmt[c] = sharpe_anual_fmt[c].map(lambda x: f"{x:.4f}")
    print("\n--- Sharpe por ano ---")
    print(sharpe_anual_fmt.to_string(index=False))
else:
    print("\n[Aviso] Não foi possível calcular Sharpe por ano (dados insuficientes).")

# Período completo
print("\n--- Período completo ---")
print(f"Sharpe (base diária): {sharpe_total_d:.4f}")
print(f"Sharpe (base mensal): {sharpe_total_m:.4f}")


=== ÍNDICE DE SHARPE (ANUALIZADO) ===
Taxa livre de risco (a.a.) considerada: 0.00%

--- Sharpe por ano ---
 Ano Sharpe (base diária) Sharpe (base mensal)
2011               0.0110               0.2226
2012               2.3261               2.1516
2013               1.0457               1.4736
2014              -0.3096              -0.3352
2015              -0.7849              -0.6786
2016               2.5996               1.9796
2017               3.9575               4.4345
2018               0.6694               0.5849
2019               2.5421               2.3691

--- Período completo ---
Sharpe (base diária): 1.4668
Sharpe (base mensal): 1.3090


# **# % MESES POSITIVOS - CARTEIRA FM**

In [9]:
# ----------- Percentual de meses positivos -----------

# Se ainda não tiver ret_mensal no script (por garantia), recalcule a partir da série diária:
if 'ret_mensal' not in globals() or ret_mensal is None:
    valor_mensal = serie_valor.resample('ME').last().dropna()
    ret_mensal = valor_mensal.pct_change().dropna()

# (1) Percentual de meses positivos no período inteiro
total_meses = len(ret_mensal)
meses_positivos = int((ret_mensal > 0).sum())
perc_positivos_total = (meses_positivos / total_meses * 100.0) if total_meses > 0 else float('nan')

print("\n=== MESES POSITIVOS ===")
print(f"Meses positivos (período): {meses_positivos} de {total_meses}  ->  {perc_positivos_total:.2f}%")

# (2) (Opcional) Percentual de meses positivos por ano
positivos_por_ano = []
for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
    n = len(sub)
    k = int((sub > 0).sum())
    perc = (k / n * 100.0) if n > 0 else float('nan')
    positivos_por_ano.append({"Ano": int(ano), "Meses Positivos": k, "Total de Meses": n, "% Meses Positivos": perc})

if positivos_por_ano:
    df_pos_ano = pd.DataFrame(positivos_por_ano).sort_values("Ano")
    # Formata a coluna percentual com 2 casas
    df_pos_ano["% Meses Positivos"] = df_pos_ano["% Meses Positivos"].map(lambda x: f"{x:.2f}%")
    print("\n--- Por ano ---")
    print(df_pos_ano.to_string(index=False))


=== MESES POSITIVOS ===
Meses positivos (período): 64 de 107  ->  59.81%

--- Por ano ---
 Ano  Meses Positivos  Total de Meses % Meses Positivos
2011                5              11            45.45%
2012               10              12            83.33%
2013                9              12            75.00%
2014                4              12            33.33%
2015                3              12            25.00%
2016                7              12            58.33%
2017               11              12            91.67%
2018                6              12            50.00%
2019                9              12            75.00%


# **# EXCEL - RETORNO MENSAL CARTEIRA FM**

In [29]:

RET_MES_FM = ret_mensal_df.copy()

arquivo_saida_FM = '0_6_1_RET_MES_FM.xlsx'
nome_aba_FM = 'RET_MES_FM'
coluna_percentual_FM = 'Retorno Mensal FM'  # confirme o nome exato no DataFrame

with pd.ExcelWriter(arquivo_saida_FM, engine='openpyxl') as writer:
    RET_MES_FM.to_excel(writer, index=False, sheet_name=nome_aba_FM)

    ws_FM = writer.sheets[nome_aba_FM]

    # Descobre o índice (1-based) da coluna alvo
    try:
        col_idx = RET_MES_FM.columns.get_loc(coluna_percentual_FM) + 1
    except KeyError:
        raise KeyError(
            f'A coluna "{coluna_percentual_FM}" não foi encontrada no DataFrame. '
            f'Colunas existentes: {list(RET_MES_FM.columns)}'
        )

    # Aplica formatação de porcentagem (duas casas) da linha 2 até a última (linha 1 é o cabeçalho)
    for col in ws_FM.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2, max_row=ws_FM.max_row):
        for cell in col:
            cell.number_format = '0.00%'

print(f"Arquivo Excel {arquivo_saida_FM} criado com sucesso!")


Arquivo Excel 0_6_1_RET_MES_FM.xlsx criado com sucesso!


In [30]:
Arq_RET_MES_FM = op.load_workbook('0_6_1_RET_MES_FM.xlsx')                                 # *Carregando arquivo Excel 8_1_RET_MES_FM.xlxl
Plan_RET_MES_FM = Arq_RET_MES_FM['RET_MES_FM']                                     # *Carregando planilha Excel expec[ifica em 8_1_RET_MES_FM.xlxl
Arq_RET_MES_FM

In [31]:
Arqler_RET_MES_FM = pd.read_excel('0_6_1_RET_MES_FM.xlsx')                                 # *Lendo arquivo Excel 8_1_RET_MES_FM.xlxl
Arqler_RET_MES_FM

,Ano,Mes,Retorno Mensal FM
0,2011,2,-0.003590
1,2011,3,0.017027
2,2011,4,-0.006506
3,2011,5,0.046174
4,2011,6,0.006201
5,2011,7,-0.042150
6,2011,8,-0.021481
7,2011,9,-0.014651
8,2011,10,0.022220
9,2011,11,-0.000135


# **# EXCEL - RETORNO ANUAL FM**

In [13]:

print('Índice:', RET_MES_FM.index.name)
print('Colunas:', list(RET_MES_FM.columns))


Índice: Date
Colunas: ['Ano', 'Mes', 'Retorno Mensal FM']


In [32]:

# === Configuração ===
RET_ANO_FM = ret_anual_df.copy()

arquivo_saida_ano_FM = '0_6_2_RET_ANO_FM.xlsx'
nome_aba_FM = 'RET_ANO_FM'
coluna_percentual_ano_FM = 'Retorno Anual_FM'
nome_coluna_data = 'Date'  # ajuste se for 'DATA', 'date', etc.

# === 1) Garanta que a coluna de data exista (se estiver no índice, traz para coluna) ===
if RET_ANO_FM.index.name == nome_coluna_data or nome_coluna_data not in RET_ANO_FM.columns:
    RET_ANO_FM = RET_ANO_FM.reset_index()

# Caso a coluna ainda não exista (ex: índice sem nome), tenta achar algo parecido
if nome_coluna_data not in RET_ANO_FM.columns:
    # Tenta detectar colunas de data por nomes comuns
    candidatos = [c for c in RET_ANO_FM.columns if str(c).lower() in ('date','data','dt','dia','dia_ref','data_ref')]
    if candidatos:
        nome_coluna_data = candidatos[0]  # usa o primeiro candidato encontrado

# === 2) Converte a coluna de data para datetime de forma robusta ===
def _converte_para_datetime_coluna(series: pd.Series) -> pd.Series:
    # Se já é datetime, retorna
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    # Se for período (ex. PeriodIndex convertido pra Series), transforma em timestamp
    if hasattr(series, 'dt') and hasattr(series.dt, 'to_timestamp'):
        try:
            return series.dt.to_timestamp()
        except Exception:
            pass

    # Numérico: pode ser ano, YYYYMMDD, Unix em s ou ms
    if pd.api.types.is_numeric_dtype(series):
        s_num = pd.to_numeric(series, errors='coerce')

        # Heurística: se todos (ou maioria) são anos plausíveis, usa 01/01/ano
        if s_num.dropna().between(1900, 2100).mean() > 0.9:
            anos = s_num.astype('Int64')
            return pd.to_datetime(anos.astype(str) + '-01-01', errors='coerce')

        # Heurística: YYYYMMDD (8 dígitos)
        mask_yyyymmdd = s_num.dropna().astype('Int64').astype(str).str.len().eq(8).mean() > 0.9
        if mask_yyyymmdd:
            return pd.to_datetime(s_num.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')

        # Heurística: Unix timestamp (segundos vs milissegundos)
        # valores típicos: segundos ~1.5e9, milissegundos ~1.5e12
        abs_mediana = s_num.dropna().abs().median()
        if abs_mediana > 1e11:   # provavelmente ms
            return pd.to_datetime(s_num, unit='ms', origin='unix', errors='coerce')
        elif abs_mediana > 1e9:  # provavelmente s
            return pd.to_datetime(s_num, unit='s', origin='unix', errors='coerce')

        # Se não bateu nada, tenta parse "genérico"
        return pd.to_datetime(series, errors='coerce')

    # Texto: tenta formatos comuns primeiro, depois genérico
    if pd.api.types.is_object_dtype(series):
        s_str = series.astype(str)

        # Tenta ISO / americano
        s_try = pd.to_datetime(s_str, errors='coerce', utc=False)
        if s_try.notna().mean() > 0.8:
            return s_try

        # Tenta BR (dd/mm/yyyy)
        s_try = pd.to_datetime(s_str, errors='coerce', dayfirst=True, utc=False)
        if s_try.notna().mean() > 0.8:
            return s_try

        # Tenta YYYY-MM-DD explícito
        try:
            s_try = pd.to_datetime(s_str, format='%Y-%m-%d', errors='coerce')
            if s_try.notna().mean() > 0.8:
                return s_try
        except Exception:
            pass

        # Tenta DD/MM/YYYY explícito
        try:
            s_try = pd.to_datetime(s_str, format='%d/%m/%Y', errors='coerce')
            if s_try.notna().mean() > 0.8:
                return s_try
        except Exception:
            pass

        # Último recurso
        return pd.to_datetime(s_str, errors='coerce')

    # Caso final
    return pd.to_datetime(series, errors='coerce')

# Aplica conversão se a coluna de data existir
if nome_coluna_data in RET_ANO_FM.columns:
    RET_ANO_FM[nome_coluna_data] = _converte_para_datetime_coluna(RET_ANO_FM[nome_coluna_data])

# === 3) Escreve SEM índice e formata ===
with pd.ExcelWriter(arquivo_saida_ano_FM, engine='openpyxl') as writer:
    RET_ANO_FM.to_excel(writer, index=False, sheet_name=nome_aba_FM)
    ws = writer.sheets[nome_aba_FM]

    # índices das colunas no Excel (1-based)
    def _pos_excel(df, colname):
        return df.columns.get_loc(colname) + 1

    # Formatação de percentual (apenas a coluna indicada)
    try:
        col_idx_pct = _pos_excel(RET_ANO_FM, coluna_percentual_ano_FM)
        for col in ws.iter_cols(min_col=col_idx_pct, max_col=col_idx_pct, min_row=2, max_row=ws.max_row):
            for cell in col:
                cell.number_format = '0.00%'
    except Exception:
        # Se a coluna não existir, apenas segue
        pass

    # Formatação de data — somente se a coluna existir e for datetime
    if (nome_coluna_data in RET_ANO_FM.columns and 
        pd.api.types.is_datetime64_any_dtype(RET_ANO_FM[nome_coluna_data])):
        col_idx_date = _pos_excel(RET_ANO_FM, nome_coluna_data)
        for col in ws.iter_cols(min_col=col_idx_date, max_col=col_idx_date, min_row=2, max_row=ws.max_row):
            for cell in col:
                cell.number_format = 'dd/mm/yyyy'

print(f"Arquivo Excel {arquivo_saida_ano_FM} criado com sucesso!")


Arquivo Excel 0_6_2_RET_ANO_FM.xlsx criado com sucesso!


In [33]:
Arq_RET_ANO_FM = op.load_workbook('0_6_2_RET_ANO_FM.xlsx')                                 # *Carregando arquivo Excel 9_2_RET_ANO_IBOV.xlxl
Plan_RET_ANO_FM = Arq_RET_ANO_FM['RET_ANO_FM']                                     # *Carregando planilha Excel expec[ifica em Ree9_2_RET_ANO_IBOV.xlxl
Arq_RET_ANO_FM

In [34]:
Arq_RET_ANO_FM = pd.read_excel('0_6_2_RET_ANO_FM.xlsx')                # *Lendo arquivo Excel 8_2_RET_ANO_FM.xlsx
Arq_RET_ANO_FM


,Date,Retorno Anual_FM
0,2012-01-01,0.220175
1,2013-01-01,0.178036
2,2014-01-01,-0.030617
3,2015-01-01,-0.092305
4,2016-01-01,0.735565
5,2017-01-01,0.572544
6,2018-01-01,0.068547
7,2019-01-01,0.774240


# **# EXCEL - QUANTIDADE DE CADA PAPEL REBALANCEADA EM CADA INÍCIO DE ANO**

In [48]:
df_resumo_qtds_ticker = df_resumo_qtds
if 'Ticker' not in df_resumo_qtds_ticker.columns:
    df_resumo_qtds_ticker = df_resumo_qtds_ticker.reset_index()

In [49]:
QUANT_PAPEL_REB_ANUAL = df_resumo_qtds_ticker 

QUANT_PAPEL_REB_ANUAL.to_excel('0_6_3_QUANT_PAPEL_REB_ANUAL.xlsx', index=False, engine='openpyxl')         # *Criando arquivo Excel 8_3_VOL_ANO_FM.xlsx
print("Arquivo Excel 0_6_3_QUANT_PAPEL_REB_ANUAL criado com sucesso!")                            # *Confirmando a criação do arquivo excel 8_3_VOL_ANO_FM.xlsx

Arquivo Excel 0_6_3_QUANT_PAPEL_REB_ANUAL criado com sucesso!


In [50]:
Arq_QUANT_PAPEL_REB_ANUAL = op.load_workbook('0_6_3_QUANT_PAPEL_REB_ANUAL.xlsx')                           # *Carregando arquivo Excel 8_3_VOL_ANO_FM.xlsx
Arq_VOL_ANO_FM = Arq_QUANT_PAPEL_REB_ANUAL['Sheet1']                                          # *Carregando planilha Excel expec[ifica em 8_3_VOL_ANO_FM.xlsx
Arq_QUANT_PAPEL_REB_ANUAL

In [51]:
ArqLer_QUANT_PAPEL_REB_ANUAL = pd.read_excel('0_6_3_QUANT_PAPEL_REB_ANUAL.xlsx')                  # *Lendo arquivo Excel 8_3_VOL_ANO_FM.xlsx
ArqLer_QUANT_PAPEL_REB_ANUAL

,Ticker,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,BMKS3.SA,1,2,2,3,3,3,3,4,5
1,GRND3.SA,148,172,93,95,103,81,125,120,142
2,BRKM5.SA,13,18,22,16,18,10,13,15,14
3,BRKM3.SA,15,22,29,22,31,18,15,15,14
4,WLMM3.SA,32,31,39,45,43,39,69,96,152
5,WLMM4.SA,28,23,22,26,23,59,174,143,150
6,PNVL3.SA,57,48,23,35,33,22,17,42,59
7,AHEB3.SA,6,12,10,14,13,13,7,5,7
8,AHEB6.SA,10,6,7,8,8,24,7,8,9
9,AHEB5.SA,5,4,7,10,10,9,4,6,6


# **# EXCEL - VALOR FINANCEIRO DE CADA PAPEL REBALANCEADA EM CADA INÍCIO DE ANO**

In [52]:
df_resumo_financeiro_ticker = df_resumo_financeiro
if 'Ticker' not in df_resumo_financeiro_ticker.columns:
    df_resumo_financeiro_ticker = df_resumo_financeiro_ticker.reset_index()

In [53]:
VALOR_PAPEL_REB_ANUAL = df_resumo_financeiro_ticker 

VALOR_PAPEL_REB_ANUAL.to_excel('0_6_4_VALOR_PAPEL_REB_ANUAL.xlsx', index=False, engine='openpyxl')         # *Criando arquivo Excel 8_3_VOL_ANO_FM.xlsx
print("Arquivo Excel 0_6_4_VALOR_PAPEL_REB_ANUAL criado com sucesso!")                            # *Confirmando a criação do arquivo excel 8_3_VOL_ANO_FM.xlsx

Arquivo Excel 0_6_4_VALOR_PAPEL_REB_ANUAL criado com sucesso!


In [54]:
Arq_VALOR_PAPEL_REB_ANUAL = op.load_workbook('0_6_4_VALOR_PAPEL_REB_ANUAL.xlsx')                           # *Carregando arquivo Excel 8_3_VOL_ANO_FM.xlsx
Arq_VALOR_PAPEL_REB_ANUAL = Arq_VALOR_PAPEL_REB_ANUAL['Sheet1']                                          # *Carregando planilha Excel expec[ifica em 8_3_VOL_ANO_FM.xlsx
Arq_VALOR_PAPEL_REB_ANUAL

<Worksheet "Sheet1">

In [55]:
ArqLer_VALOR_PAPEL_REB_ANUAL = pd.read_excel('0_6_4_VALOR_PAPEL_REB_ANUAL.xlsx')                  # *Lendo arquivo Excel 8_3_VOL_ANO_FM.xlsx
ArqLer_VALOR_PAPEL_REB_ANUAL

,Ticker,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,BMKS3.SA,161.604324,155.934006,211.416718,291.236115,259.713982,226.980721,322.819267,604.270813,644.825745
1,GRND3.SA,142.924822,144.493989,178.651271,206.955637,198.360478,181.561337,317.229867,507.578259,541.004249
2,BRKM5.SA,147.421180,143.311689,181.897818,198.239700,197.619804,179.940166,326.170845,499.367695,559.596626
3,BRKM3.SA,145.044637,146.875097,176.197120,206.236132,198.773691,183.269514,324.263105,496.124439,530.261963
4,WLMM3.SA,146.203964,141.635090,178.186081,205.599325,196.461577,178.186081,315.252298,507.145020,542.553812
5,WLMM4.SA,147.028505,143.160839,176.403503,204.591656,195.573700,180.239771,315.728140,507.806143,540.106130
6,PNVL3.SA,144.588721,144.314529,176.009478,201.742651,198.283210,174.768761,313.123173,501.357056,534.237422
7,AHEB3.SA,155.347092,142.031616,192.334499,201.951218,187.526131,187.429961,310.694187,503.028679,522.406242
8,AHEB6.SA,148.445177,155.246464,181.172611,207.054413,207.054413,177.424507,310.492928,473.132050,532.273556
9,AHEB5.SA,142.725382,137.803818,179.144968,196.764164,196.961021,177.264919,311.003540,564.661011,564.661011


# **# EXCEL - VOLATILIDADE ANUAL FM - MÉDIA DIÁRIA/MENSAL**

In [56]:
VOL_ANO_FM = vol_anual_fmt

VOL_ANO_FM.to_excel('0_6_5_VOL_ANO_FM.xlsx', index=False, engine='openpyxl')         # *Criando arquivo Excel 8_3_VOL_ANO_FM.xlsx
print("Arquivo Excel 0_6_5_VOL_ANO_FM criado com sucesso!")                            # *Confirmando a criação do arquivo excel 8_3_VOL_ANO_FM.xlsx

Arquivo Excel 0_6_5_VOL_ANO_FM criado com sucesso!


In [57]:
Arq_VOL_ANO_FM = op.load_workbook('0_6_5_VOL_ANO_FM.xlsx')                           # *Carregando arquivo Excel 8_3_VOL_ANO_FM.xlsx
Arq_VOL_ANO_FM = Arq_VOL_ANO_FM['Sheet1']                                          # *Carregando planilha Excel expec[ifica em 8_3_VOL_ANO_FM.xlsx
Arq_VOL_ANO_FM

<Worksheet "Sheet1">

In [58]:
ArqLer_Arq_VOL_ANO_D_FM = pd.read_excel('0_6_5_VOL_ANO_FM.xlsx')                  # *Lendo arquivo Excel 8_3_VOL_ANO_FM.xlsx
ArqLer_Arq_VOL_ANO_D_FM

,Ano,Vol Anualizada (base diária),Vol Anualizada (base mensal)
0,2011,11.29%,8.21%
1,2012,8.98%,9.52%
2,2013,17.28%,11.61%
3,2014,8.93%,8.32%
4,2015,11.76%,13.08%
5,2016,22.40%,30.43%
6,2017,11.53%,10.51%
7,2018,11.06%,12.60%
8,2019,24.02%,25.96%


# **# EXCEL - DRAWDOWN MÁXIMO FM**

In [59]:
MDD_FM = mdd_por_ano

MDD_FM.to_excel('0_6_6_MDD_FM.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 8_4_MDD_FM.xlsx
print("Arquivo Excel 0_6_6_MDD_FM criado com sucesso!")                              # *Confirmando a criação do arquivo excel 8_4_MDD_FM.xlsx

Arquivo Excel 0_6_6_MDD_FM criado com sucesso!


In [60]:
Arq_MDD_FM = op.load_workbook('0_6_6_MDD_FM.xlsx')                                # *Carregando arquivo Excel 8_4_MDD_FM.xlsx
Arq_MDD_FM = Arq_MDD_FM['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 8_4_MDD_FM.xlsx
Arq_MDD_FM

<Worksheet "Sheet1">

In [61]:
ArqLer_MDD_FM = pd.read_excel('0_6_6_MDD_FM.xlsx')                                # *Lendo arquivo Excel 8_4_MDD_FM.xlsx
ArqLer_MDD_FM

,Ano,MDD,Peak,Trough
0,2011,-0.133547,2011-07-04,2011-08-08
1,2012,-0.071232,2012-05-02,2012-06-11
2,2013,-0.120964,2013-05-08,2013-07-10
3,2014,-0.098626,2014-07-08,2014-12-15
4,2015,-0.157389,2015-05-04,2015-09-29
5,2016,-0.071360,2016-01-05,2016-01-26
6,2017,-0.043201,2017-05-16,2017-05-22
7,2018,-0.069165,2018-05-10,2018-06-19
8,2019,-0.081217,2019-09-24,2019-10-07


# **# EXCEL - ÍNDICE SHARPE FM**

In [62]:
SHARPE_FM = sharpe_anual
SHARPE_FM.to_excel('0_6_7_SHARPE_FM.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 8_5_SHARPE_FM.xlsx
print("Arquivo Excel 0_6_7_SHARPE_FM criado com sucesso!")                               # *Confirmando a criação do arquivo excel 8_5_SHARPE_FM.xlsx

Arquivo Excel 0_6_7_SHARPE_FM criado com sucesso!


In [63]:
Arq_SHARPE_FM = op.load_workbook('0_6_7_SHARPE_FM.xlsx')                                # *Carregando arquivo Excel 8_5_SHARPE_FM.xlsx
Arq_SHARPE_FM = Arq_SHARPE_FM['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 8_5_SHARPE_FM.xlsx
Arq_SHARPE_FM

<Worksheet "Sheet1">

In [65]:
ArqLer_SHARPE_FM = pd.read_excel('0_6_7_SHARPE_FM.xlsx')                                # *Lendo arquivo Excel 8_5_SHARPE_FM.xlsx
ArqLer_SHARPE_FM

,Ano,Sharpe (base diária),Sharpe (base mensal)
0,2011,0.010957,0.222637
1,2012,2.326059,2.151578
2,2013,1.045701,1.473557
3,2014,-0.309583,-0.335204
4,2015,-0.784944,-0.678615
5,2016,2.599601,1.979554
6,2017,3.957521,4.434525
7,2018,0.669430,0.584877
8,2019,2.542074,2.369050


# **# EXCEL - % MESES POSITIVOS FM**

In [66]:
POSITIVOS_ANO_FM = df_pos_ano

POSITIVOS_ANO_FM = POSITIVOS_ANO_FM.copy()
POSITIVOS_ANO_FM.to_excel('0_6_8_POSITIVOS_ANO_FM.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 8_6_POSITIVOS_ANO.xlsx
print("Arquivo Excel 0_6_8_POSITIVOS_ANO_FM criado com sucesso!")  

Arquivo Excel 0_6_8_POSITIVOS_ANO_FM criado com sucesso!


In [68]:
Arq_POSITIVOS_ANO_FM = op.load_workbook('0_6_8_POSITIVOS_ANO_FM.xlsx')                                # *Carregando arquivo Excel 8_6_POSITIVOS_ANO_FM.xlsx
Arq_POSITIVOS_ANO_FM = Arq_POSITIVOS_ANO_FM['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 8_6_POSITIVOS_ANO_FM.xlsx
Arq_POSITIVOS_ANO_FM

<Worksheet "Sheet1">

In [69]:
ArqLer_POSITIVOS_ANO_FM = pd.read_excel('0_6_8_POSITIVOS_ANO_FM.xlsx')                                # *Lendo arquivo Excel 8_6_POSITIVOS_ANO_FM.xlsx
ArqLer_POSITIVOS_ANO_FM

,Ano,Meses Positivos,Total de Meses,% Meses Positivos
0,2011,5,11,45.45%
1,2012,10,12,83.33%
2,2013,9,12,75.00%
3,2014,4,12,33.33%
4,2015,3,12,25.00%
5,2016,7,12,58.33%
6,2017,11,12,91.67%
7,2018,6,12,50.00%
8,2019,9,12,75.00%
